In [1]:
import pandas as pd

In [ ]:
# Reading our training data
train_normal = pd.read_csv("dataset/TEP_FaultFree_Training.csv")
train_faulty = pd.read_csv("dataset/TEP_Faulty_Training.csv")

In [3]:
# UNDERSTANDING THE DATA
'''
print("Normal shape:", train_normal.shape)
print("Faulty shape:", train_faulty.shape)

print("\nColumns:")
print(train_normal.columns.tolist())

print("\nNumber of normal simulation runs:")
print(train_normal["simulationRun"].nunique())

print("\nNumber of faulty simulation runs:")
print(train_faulty["simulationRun"].nunique())

print("\nFault types:")
print(train_faulty["faultNumber"].value_counts().sort_index())

print("\nSamples per normal simulation run:")
print(train_normal.groupby("simulationRun")["sample"].count().describe())

# Basic information about the data

print("Normal fault numbers:")
print(train_normal["faultNumber"].unique())

print("\nFaulty fault numbers:")
print(train_faulty["faultNumber"].unique())

print("\nFaulty runs for fault 1:")
print(train_faulty[train_faulty["faultNumber"] == 1]["simulationRun"].unique()[:20])

print("\nFaulty runs for fault 2:")
print(train_faulty[train_faulty["faultNumber"] == 2]["simulationRun"].unique()[:20])

print("\nSamples for fault 1, run 1:")
print(
    train_faulty[
        (train_faulty["faultNumber"] == 1) &
        (train_faulty["simulationRun"] == 1)
    ]["sample"].values
)

print("\nMissing values:")
print(train_normal.isna().sum().sum())
print(train_faulty.isna().sum().sum())

print("\nData types:")
print(train_normal.dtypes.value_counts())

'''

'\nprint("Normal shape:", train_normal.shape)\nprint("Faulty shape:", train_faulty.shape)\n\nprint("\nColumns:")\nprint(train_normal.columns.tolist())\n\nprint("\nNumber of normal simulation runs:")\nprint(train_normal["simulationRun"].nunique())\n\nprint("\nNumber of faulty simulation runs:")\nprint(train_faulty["simulationRun"].nunique())\n\nprint("\nFault types:")\nprint(train_faulty["faultNumber"].value_counts().sort_index())\n\nprint("\nSamples per normal simulation run:")\nprint(train_normal.groupby("simulationRun")["sample"].count().describe())\n\n# Basic information about the data\n\nprint("Normal fault numbers:")\nprint(train_normal["faultNumber"].unique())\n\nprint("\nFaulty fault numbers:")\nprint(train_faulty["faultNumber"].unique())\n\nprint("\nFaulty runs for fault 1:")\nprint(train_faulty[train_faulty["faultNumber"] == 1]["simulationRun"].unique()[:20])\n\nprint("\nFaulty runs for fault 2:")\nprint(train_faulty[train_faulty["faultNumber"] == 2]["simulationRun"].unique()[

In [4]:
## DEFINING THE FEATURES
feature_cols = [
    col for col in train_normal.columns
    if col.startswith("xmeas_") or col.startswith("xmv_")
] # Get all the feature columns

print("Number of features:", len(feature_cols))

# Add the binary 0,1 for fault free and faulty data

train_normal["target"] = 0 # Fault-free data


train_faulty["target"] = (
    train_faulty["sample"] >= 20
).astype(int) # Faulty data

# See the number of normal and faulty data points we have
print(train_normal["target"].value_counts())
print(train_faulty["target"].value_counts())

# Vertically stack both of them to act as our dataset
train_detection = pd.concat(
    [train_normal, train_faulty],
    ignore_index=True
)

print(train_detection.shape)
print(train_detection["target"].value_counts())

Number of features: 52
target
0    250000
Name: count, dtype: int64
target
1    4810000
0     190000
Name: count, dtype: int64
(5250000, 56)
target
1    4810000
0     440000
Name: count, dtype: int64


In [5]:
# See how many normal and faulty runs there are
normal_runs = train_normal["simulationRun"].unique()
print(len(normal_runs))

faulty_runs = train_faulty[
    ["faultNumber", "simulationRun"]
].drop_duplicates()

print(faulty_runs.shape)

500
(10000, 2)


In [6]:
from sklearn.model_selection import train_test_split

# Doing the split on simulation numbers. normal train would contain 400 numbers and validation would contain 100
normal_train_runs, normal_val_runs = train_test_split(
    normal_runs,
    test_size=0.2,
    random_state=42
)

faulty_train_runs, faulty_val_runs = train_test_split(
    faulty_runs,
    test_size=0.2,
    random_state=42
)

In [7]:
# The actual data splits
normal_train = train_normal[
    train_normal["simulationRun"].isin(normal_train_runs)
]

normal_val = train_normal[
    train_normal["simulationRun"].isin(normal_val_runs)
]

faulty_train = train_faulty.merge(
    faulty_train_runs,
    on=["faultNumber", "simulationRun"]
)

faulty_val = train_faulty.merge(
    faulty_val_runs,
    on=["faultNumber", "simulationRun"]
)

# Stack them to get the data we will perform our fits on. Still need to do the preprocessing
train_detection = pd.concat(
    [normal_train, faulty_train],
    ignore_index=True
)

val_detection = pd.concat(
    [normal_val, faulty_val],
    ignore_index=True
)

In [8]:
# Checking the shapes and all of the sets we created
'''
print("Training shape:", train_detection.shape)
print("Validation shape:", val_detection.shape)

print("\nTraining targets:")
print(train_detection["target"].value_counts())

print("\nValidation targets:")
print(val_detection["target"].value_counts())

print(
    set(normal_train["simulationRun"])
    & set(normal_val["simulationRun"])
)

train_faulty_run_ids = set(
    zip(
        faulty_train["faultNumber"],
        faulty_train["simulationRun"]
    )
)

val_faulty_run_ids = set(
    zip(
        faulty_val["faultNumber"],
        faulty_val["simulationRun"]
    )
)

print(train_faulty_run_ids & val_faulty_run_ids)
'''

'\nprint("Training shape:", train_detection.shape)\nprint("Validation shape:", val_detection.shape)\n\nprint("\nTraining targets:")\nprint(train_detection["target"].value_counts())\n\nprint("\nValidation targets:")\nprint(val_detection["target"].value_counts())\n\nprint(\n    set(normal_train["simulationRun"])\n    & set(normal_val["simulationRun"])\n)\n\ntrain_faulty_run_ids = set(\n    zip(\n        faulty_train["faultNumber"],\n        faulty_train["simulationRun"]\n    )\n)\n\nval_faulty_run_ids = set(\n    zip(\n        faulty_val["faultNumber"],\n        faulty_val["simulationRun"]\n    )\n)\n\nprint(train_faulty_run_ids & val_faulty_run_ids)\n'

In [9]:
# Feature List
feature_cols = [
    col for col in train_detection.columns
    if col.startswith("xmeas_") or col.startswith("xmv_")
]

print(len(feature_cols))

# Create X and y

X_train = train_detection[feature_cols]
y_train = train_detection["target"] 

X_val = val_detection[feature_cols]
y_val = val_detection["target"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

52
X_train: (4200000, 52)
y_train: (4200000,)
X_val: (1050000, 52)
y_val: (1050000,)


In [10]:
# Scaling, Standardizing
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)



Precision: 0.9161904761904762
Recall: 1.0
F1: 0.9562624254473161
PR-AUC: 0.9161904761904762

Confusion matrix:
[[     0  88000]
 [     0 962000]]


## Majority Class Classifier

In [ ]:
# Always predict the majority class
y_pred_baseline = [1] * len(y_val)

print("Precision:", precision_score(y_val, y_pred_baseline))
print("Recall:", recall_score(y_val, y_pred_baseline))
print("F1:", f1_score(y_val, y_pred_baseline))
print("PR-AUC:", average_precision_score(y_val, y_pred_baseline))

print("\nConfusion matrix:")
print(confusion_matrix(y_val, y_pred_baseline))

## Logistic Regression

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model_lr = LogisticRegression(
    max_iter=1000,
    class_weight=None,
    random_state=42
)

model_lr.fit(X_train_scaled, y_train)

y_pred_lr = model_lr.predict(X_val_scaled)

print(classification_report(y_val, y_pred_lr))
print("Confusion matrix:")
print(confusion_matrix(y_val, y_pred_lr))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     88000
           1       0.92      1.00      0.96    962000

    accuracy                           0.92   1050000
   macro avg       0.46      0.50      0.48   1050000
weighted avg       0.84      0.92      0.88   1050000

Confusion matrix:
[[     0  88000]
 [   210 961790]]


In [13]:
from sklearn.metrics import confusion_matrix
import numpy as np

y_prob_lr = model_lr.predict_proba(X_val_scaled)[:, 1]

print("Minimum probability:", y_prob_lr.min())
print("Maximum probability:", y_prob_lr.max())
print("Probability percentiles:")
print(np.percentile(y_prob_lr, [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100]))

Minimum probability: 0.29460529334262925
Maximum probability: 1.0
Probability percentiles:
[0.29460529 0.70401163 0.78573721 0.81980458 0.86757358 0.91866679
 0.98183324 0.99704376 1.         1.         1.        ]


In [14]:
from sklearn.metrics import precision_score, recall_score, f1_score

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
    y_pred = (y_prob_lr >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print("Precision:", precision_score(y_val, y_pred, zero_division=0))
    print("Recall:", recall_score(y_val, y_pred, zero_division=0))
    print("F1:", f1_score(y_val, y_pred, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_val, y_pred))


Threshold: 0.1
Precision: 0.9161904761904762
Recall: 1.0
F1: 0.9562624254473161
Confusion matrix:
[[     0  88000]
 [     0 962000]]

Threshold: 0.2
Precision: 0.9161904761904762
Recall: 1.0
F1: 0.9562624254473161
Confusion matrix:
[[     0  88000]
 [     0 962000]]

Threshold: 0.3
Precision: 0.9161903963718061
Recall: 0.9999989604989605
F1: 0.9562619066908085
Confusion matrix:
[[     0  88000]
 [     1 961999]]

Threshold: 0.4
Precision: 0.9161880815642351
Recall: 0.999968814968815
F1: 0.9562468625277712
Confusion matrix:
[[     0  88000]
 [    30 961970]]

Threshold: 0.5
Precision: 0.9161737109326628
Recall: 0.9997817047817048
F1: 0.9561534752633227
Confusion matrix:
[[     0  88000]
 [   210 961790]]


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    average_precision_score
)

model_lr_balanced = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model_lr_balanced.fit(X_train_scaled, y_train)

y_pred_balanced = model_lr_balanced.predict(X_val_scaled)
y_prob_balanced = model_lr_balanced.predict_proba(X_val_scaled)[:, 1]

print(classification_report(y_val, y_pred_balanced))
print("Confusion matrix:")
print(confusion_matrix(y_val, y_pred_balanced))
print("PR-AUC:", average_precision_score(y_val, y_prob_balanced))

              precision    recall  f1-score   support

           0       0.16      0.87      0.27     88000
           1       0.98      0.58      0.73    962000

    accuracy                           0.60   1050000
   macro avg       0.57      0.73      0.50   1050000
weighted avg       0.91      0.60      0.69   1050000

Confusion matrix:
[[ 76984  11016]
 [405515 556485]]
PR-AUC: 0.9731753189272514


In [16]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    y_pred = (y_prob_balanced >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

    print(f"\nThreshold: {threshold}")
    print("Normal recall:", tn / (tn + fp))
    print("Fault recall:", tp / (tp + fn))
    print("Fault precision:", precision_score(y_val, y_pred, zero_division=0))
    print("Fault F1:", f1_score(y_val, y_pred, zero_division=0))


Threshold: 0.1
Normal recall: 0.0003181818181818182
Fault recall: 0.9911891891891892
Fault precision: 0.9155330409334266
Fault F1: 0.951860148460491

Threshold: 0.2
Normal recall: 0.04425
Fault recall: 0.9420363825363826
Fault precision: 0.9150740398547981
Fault F1: 0.9283594856441869

Threshold: 0.3
Normal recall: 0.27682954545454547
Fault recall: 0.8363076923076923
Fault precision: 0.9266972828960327
Fault F1: 0.8791853421026605

Threshold: 0.4
Normal recall: 0.6264772727272727
Fault recall: 0.6989740124740125
Fault precision: 0.9533945947938629
Fault F1: 0.806597320310949

Threshold: 0.5
Normal recall: 0.8748181818181818
Fault recall: 0.5784667359667359
Fault precision: 0.9805885804606511
Fault F1: 0.7276686971763994

Threshold: 0.6
Normal recall: 0.9761931818181818
Fault recall: 0.490475051975052
Fault precision: 0.9955795346167805
Fault F1: 0.6571857163152572

Threshold: 0.7
Normal recall: 0.9982045454545454
Fault recall: 0.42233783783783785
Fault precision: 0.999611265429441
Fau